In [1]:
import torch
import torch.nn as nn

In [2]:
class LSTM(nn.Module):
  def __init__(self, input_dim, hidden_dim):
    super().__init__()
    self.lin = nn.Linear(input_dim+hidden_dim, hidden_dim*4)
    self.h = hidden_dim

  def forward(self, x):
    batch_size, seq_len, input_dim = x.size()

    h = torch.zeros(batch_size, self.h)
    C = torch.zeros(batch_size, self.h)

    outputs = []

    for t in range(seq_len):
      combined = torch.cat([x[:, t, :], h], dim=1)
      gates = self.lin(combined)

      f, i, g, o = torch.chunk(gates, 4, dim=1)

      f = torch.sigmoid(f)
      i = torch.sigmoid(i)
      g = torch.tanh(g)
      o = torch.sigmoid(o)

      C = g * i + C * f
      h = o * torch.tanh(C)

      outputs.append(h)

    return torch.stack(outputs, dim=1), (h, C)
